# Eval checkpoints sans YAML
Entrer manuellement :
- `DATA_DIR` : chemin du graphe DGL (Mesh8, Multimesh, ...)
- `DYNAMIC_DIR` : liste de fichiers `.pkl` à évaluer (test set)
- `CKPT_PATH_STATS` : dossier contenant `node_stats.json` / `edge_stats.json` (créés au train)
- Paramètres du modèle (mp_layers, dims...)
- Liste de checkpoints `.mdlus`

Le notebook calcule MSE (h,u,v) et CSI / CSI_over aux horizons 30 min / 3 h / 6 h / 12 h (1/6/12/24 pas) et trace l'évolution vs checkpoints.

In [1]:
import os, json, math
import torch, dgl
import numpy as np
import matplotlib.pyplot as plt
from typing import List
import sys

# ajoute le repo au PYTHONPATH
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

from python.create_dgl_dataset import TelemacDataset
from python.CustomMeshGraphNet import MeshGraphNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [2]:
# =====================
# Paramètres utilisateur
# =====================
DATA_DIR = "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Mesh8_base.bin"
DYNAMIC_DIR = [
    # ajoute ici tes .pkl de test
]
CKPT_PATH_STATS = "/work/m24046/m24046mrcr/paper/Experience1/Seed0/"  # contient node_stats.json / edge_stats.json

# Paramètres modèle
NUM_INPUT_FEATURES = 9
NUM_EDGE_FEATURES = 3
NUM_OUTPUT_FEATURES = 3
MP_LAYERS = 10
DO_CONCAT_TRICK = True
NUM_PROCESSOR_CHECKPOINT_SEGMENTS = 0

# Checkpoints à évaluer
CHECKPOINTS = [
    # ex: "/work/.../MeshGraphNet.0.40.mdlus",
]

HORIZONS_STEPS = [1, 6, 12, 24]  # 30 min, 3h, 6h, 12h
MAX_SEQUENCES = 5  # nb de séquences à moyenner
THRESHOLD_M = 0.05  # seuil CSI

In [3]:
# =====================
# Fonctions utilitaires
# =====================
def build_model():
    return MeshGraphNet(
        NUM_INPUT_FEATURES,
        NUM_EDGE_FEATURES,
        NUM_OUTPUT_FEATURES,
        processor_size=MP_LAYERS,
        hidden_dim_processor=64,
        hidden_dim_node_encoder=64,
        hidden_dim_edge_encoder=64,
        hidden_dim_node_decoder=64,
        do_concat_trick=DO_CONCAT_TRICK,
        num_processor_checkpoint_segments=NUM_PROCESSOR_CHECKPOINT_SEGMENTS,
    )

def load_state_dict(model, ckpt_path):
    state = torch.load(ckpt_path, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model

def build_dataset(sequence_length, overlap=1, split="test"):
    ds = TelemacDataset(
        name=f"eval_{split}",
        data_dir=DATA_DIR,
        dynamic_data_files=DYNAMIC_DIR,
        split=split,
        ckpt_path=CKPT_PATH_STATS,
        normalize=True,
        sequence_length=sequence_length,
        overlap=overlap,
    )
    return ds

def _denorm(xn, mean, std):
    return xn * std + mean

def _renorm(x, mean, std):
    return (x - mean) / (std + 1e-12)

def csi_from_binary(pred_mask, gt_mask):
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    denom = tp + fp + fn
    return float(tp / denom) if denom > 0 else math.nan

def rollout_sequence(model, graphs, stats, horizons_steps, threshold=0.05):
    dyn_start = graphs[0].ndata['x'].shape[1] - 3
    mx = torch.tensor([stats['h'].item(), stats['u'].item(), stats['v'].item()], device=device)
    sx = torch.tensor([stats['h_std'].item(), stats['u_std'].item(), stats['v_std'].item()], device=device)
    dy_mean = torch.tensor([stats['delta_h'].item(), stats['delta_u'].item(), stats['delta_v'].item()], device=device)
    dy_std  = torch.tensor([stats['delta_h_std'].item(), stats['delta_u_std'].item(), stats['delta_v_std'].item()], device=device)

    horizons_steps = sorted(horizons_steps)
    max_h = horizons_steps[-1]
    L = len(graphs)
    if L <= max_h:
        return {}

    g = graphs[0].to(device)
    static_part = g.ndata['x'][:, :dyn_start]
    xn_t = g.ndata['x'][:, dyn_start:dyn_start+3]
    h0_gt = _denorm(xn_t, mx, sx)[:, 0].detach().cpu().numpy()
    wet0_mask = h0_gt >= threshold

    preds_cache = {}
    with torch.no_grad():
        for t in range(max_h):
            y_pred_n = model(g.ndata['x'], g.edata['x'], g)
            x_t = _denorm(xn_t, mx, sx)
            y_pred = _denorm(y_pred_n, dy_mean, dy_std)
            x_t1 = x_t + y_pred
            xn_t1 = _renorm(x_t1, mx, sx)
            combined = torch.cat([static_part, xn_t1], dim=1)
            g = g.clone()
            g.ndata['x'] = combined
            xn_t = xn_t1
            if (t+1) in horizons_steps:
                preds_cache[t+1] = x_t1.detach().cpu()

    results = {}
    for h in horizons_steps:
        if h >= L:
            continue
        gt_graph = graphs[h]
        x_gt_n = gt_graph.ndata['x'][:, dyn_start:dyn_start+3]
        x_gt = _denorm(x_gt_n, mx, sx).cpu()
        x_pred = preds_cache[h]
        mse = torch.mean((x_pred - x_gt)**2, dim=0)  # [3]

        h_pred = x_pred[:, 0].numpy()
        h_gt = x_gt[:, 0].numpy()
        mask_all = np.ones_like(h_gt, dtype=bool)
        mask_over = ~wet0_mask
        csi_all = csi_from_binary(h_pred >= threshold, h_gt >= threshold)
        csi_over = csi_from_binary(h_pred[mask_over] >= threshold, h_gt[mask_over] >= threshold) if mask_over.any() else math.nan

        results[h] = {
            "mse": mse.tolist(),
            "csi": csi_all,
            "csi_over": csi_over,
        }
    return results

def evaluate_checkpoints(checkpoints: List[str], horizons_steps=[1,6,12,24], max_sequences=10):
    ds = build_dataset(sequence_length=max(horizons_steps)+1, overlap=1, split="test")
    stats = ds.node_stats
    metrics_per_ckpt = {}
    for ckpt in checkpoints:
        model = build_model()
        load_state_dict(model, ckpt)
        agg = {h: {"mse": [], "csi": [], "csi_over": []} for h in horizons_steps}
        nseq = min(max_sequences, len(ds))
        for idx in range(nseq):
            seq_graphs = ds[idx]
            res = rollout_sequence(model, seq_graphs, stats, horizons_steps, threshold=THRESHOLD_M)
            for h, vals in res.items():
                agg[h]["mse"].append(vals["mse"])
                agg[h]["csi"].append(vals["csi"])
                agg[h]["csi_over"].append(vals["csi_over"])
        summary = {}
        for h in horizons_steps:
            if len(agg[h]["mse"]) == 0:
                continue
            summary[h] = {
                "mse_mean": np.nanmean(np.stack(agg[h]["mse"], axis=0), axis=0).tolist(),
                "csi_mean": float(np.nanmean(agg[h]["csi"])),
                "csi_over_mean": float(np.nanmean(agg[h]["csi_over"])),
            }
        metrics_per_ckpt[ckpt] = summary
    return metrics_per_ckpt


In [4]:
# =====================
# Evaluation
# =====================
results = evaluate_checkpoints(CHECKPOINTS, horizons_steps=HORIZONS_STEPS, max_sequences=MAX_SEQUENCES)
results

In [5]:
# =====================
# Traces rapides (CSI / CSI_over / MSE_h)
# =====================
if results:
    ckpt_labels = list(results.keys())
    # essai de parser un "epoch" dans le nom (sinon index)
    def parse_epoch(label):
        for token in os.path.basename(label).replace('.', ' ').split():
            if token.isdigit():
                return int(token)
        return None
    xs = [parse_epoch(c) for c in ckpt_labels]
    if any(x is None for x in xs):
        xs = list(range(len(ckpt_labels)))

    fig, axes = plt.subplots(1, 3, figsize=(15,4))
    for h in HORIZONS_STEPS:
        ys_csi = [results[c].get(h, {}).get('csi_mean', np.nan) for c in ckpt_labels]
        ys_over = [results[c].get(h, {}).get('csi_over_mean', np.nan) for c in ckpt_labels]
        axes[0].plot(xs, ys_csi, marker='o', label=f"CSI h={h}")
        axes[1].plot(xs, ys_over, marker='o', label=f"CSI_over h={h}")
        # mse sur h (index 0)
        ys_mse_h = [results[c].get(h, {}).get('mse_mean', [np.nan])[0] for c in ckpt_labels]
        axes[2].plot(xs, ys_mse_h, marker='o', label=f"MSE_h h={h}")

    axes[0].set_title("CSI vs checkpoint")
    axes[1].set_title("CSI_over vs checkpoint")
    axes[2].set_title("MSE(h) vs checkpoint")
    for ax in axes:
        ax.set_xlabel("checkpoint (epoch ou index)")
        ax.legend()
        ax.grid(True)
    plt.tight_layout()
else:
    print("Pas de résultats (vérifie chemins/CKPT)")